# Check extra history and forecasts

Review assigned to Viviana Pajic; review pending. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Check the supplied daily files

The models need daily totals, not millions of raw rows. Added 2022 history gives the expanded model more past examples. Checksums keep the data version fixed.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
import hashlib
sources = [ROOT / 'data/citibike_2022_daily.csv', ROOT / 'data/citibike_development_daily.csv']
display(pd.DataFrame([{'file': p.name, 'rows': len(pd.read_csv(p)),
                      'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in sources]))
audit = json.loads((ROOT / 'audit/citibike2022/audit.json').read_text())
display({k: audit[k] for k in ['days', 'counted_rides', 'boundary_carryover_rides_from_january2023']})

,file,rows,sha256
0,citibike_2022_daily.csv,365,79ca554352e67a15016aae2e418641f6b0b347a4b7fb54...
1,citibike_development_daily.csv,273,aef21b6e5c8343ef7e187a6bbda4e0dd4fc153173a3cee...


{'days': 365,
 'counted_rides': 29838442,
 'boundary_carryover_rides_from_january2023': 276}

## Check forecast inputs

Use archived forecasts with nominal 48-hour leads, not observed target-day weather. Convert real hours to New York dates. Tiny negative rain values were set to zero; raw values and flags remain.

In [2]:
from weather_search import build_weather_frame, BASE_WEATHER
weather = build_weather_frame()
display(weather[BASE_WEATHER].head())
display(weather[BASE_WEATHER].isna().sum())
print('The six fields must be available before each prediction.')
print('Exact historical publication timestamps were not verified.')

,forecast_temp_day2_mean,forecast_temp_day2_min,forecast_temp_day2_max,forecast_jma_day2_precip_sum,forecast_jma_day2_wind_mean,forecast_jma_day2_wind_max
date,,,,,,
2023-01-01,9.879167,7.3,12.0,2.6,14.662500,18.9
2023-01-02,8.229167,6.0,11.0,0.0,9.012500,13.8
2023-01-03,8.925000,7.4,10.8,1.8,13.570833,20.6
2023-01-04,11.425000,10.5,12.8,1.6,15.433333,22.0
2023-01-05,10.104167,7.7,12.9,0.2,11.541667,22.8


forecast_temp_day2_mean         0
forecast_temp_day2_min          0
forecast_temp_day2_max          0
forecast_jma_day2_precip_sum    0
forecast_jma_day2_wind_mean     0
forecast_jma_day2_wind_max      0
dtype: int64

The six fields must be available before each prediction.
Exact historical publication timestamps were not verified.


## Recreate only when needed

Small snapshots are included. These optional scripts recreate sources with code. Trip downloads are large. They are not required just to review the saved models.

In [3]:
REBUILD_SOURCES = False
if REBUILD_SOURCES:
    import subprocess
    for script in ['acquire_citibike_2022.py', 'download_forecast_temperature.py', 'download_forecast_jma.py']:
        subprocess.run([sys.executable, str(ROOT / 'scripts' / script)], cwd=ROOT, check=True)
    print('For expanded forecasts, rerun both weather scripts with --year 2022, then run merge_forecast_history.py.')
else:
    print('Using the checked daily snapshots. No raw sources downloaded.')

Using the checked daily snapshots. No raw sources downloaded.
